# 00 — フォールド割り当て生成

**1回だけ実行する。** データセット全症例を5分割し、`folds.json` としてDriveに保存する。
以降の学習ジョブはこのファイルを参照してフォールドを決定する。

In [ ]:
DRIVE_DATA_DIR   = "/content/drive/MyDrive/anglist_data"          # *_image.npy が入ったフォルダ
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/anglist_learning_curve" # folds.json の保存先
N_FOLDS = 5
RANDOM_SEED = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, random

# 全症例IDを収集
case_ids = sorted(
    f.replace("_image.npy", "")
    for f in os.listdir(DRIVE_DATA_DIR)
    if f.endswith("_image.npy")
)
print(f"症例数: {len(case_ids)}")

# シャッフルして5分割
random.seed(RANDOM_SEED)
shuffled = case_ids.copy()
random.shuffle(shuffled)

folds = {}
for k in range(N_FOLDS):
    folds[str(k + 1)] = shuffled[k::N_FOLDS]

for k, ids in folds.items():
    print(f"Fold {k}: {len(ids)} 症例")

# 保存
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(DRIVE_OUTPUT_DIR, "folds.json")
with open(out_path, "w") as f:
    json.dump({"folds": folds, "seed": RANDOM_SEED, "n_folds": N_FOLDS, "total": len(case_ids)}, f, indent=2)
print(f"\n保存完了: {out_path}")